# 08 · Launch Content Agents (AI5)

**Question:** Can a team of cooperating agents draft TARU's launch content so that ≥ 90% of it passes the claims guardrail within 2 revisions, and the author rates ≥ 80% of it at 4 of 5 or better for brand voice?

**Agents:** brief → copywriter → checks (brand rules + the AI4 guardrail, called as a tool on every claim sentence) → revise (max 2) → brand-voice critic → **your approval queue**. Autonomy level 2: the agents draft; you approve every public line.

**Outputs:** 12 social posts, 6 product descriptions, 3 launch emails (`ai/eval/content_requests.csv`). These are test outputs to prove the system works, not a campaign.

> TARU is a fictional brand created for an independent student portfolio project.

**Before running:** Runtime → Change runtime type → **T4 GPU**. The full run takes about 30–45 minutes; keep the tab open.

## 1. Setup

In [ ]:
GITHUB_USER = "anvesha2402"
DRIVE_OUT = "/content/drive/MyDrive/taru-category-launch/ai_outputs"
import os, sys
if not os.path.exists("/content/taru-category-launch"):
    !git clone -q https://github.com/{GITHUB_USER}/taru-category-launch.git /content/taru-category-launch
%cd /content/taru-category-launch
!pip install -q rank_bm25 chromadb sentence-transformers pypdf scikit-learn transformers accelerate bitsandbytes
sys.path.insert(0, "/content/taru-category-launch")
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted:", e); DRIVE_OUT = "/content/ai_outputs"
os.makedirs(DRIVE_OUT, exist_ok=True)

## 2. Load the model once (shared by all agents and the guardrail)

In [ ]:
import importlib, torch, pandas as pd
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
from ai import guardrail as g, content_agents as ca
importlib.reload(g); importlib.reload(ca)
MODEL = "Qwen/Qwen2.5-7B-Instruct"
llm = g.make_llm("local", MODEL)
print("GPU:", torch.cuda.get_device_name(0), "· model loaded")

## 3. Build the team
The guardrail is the AI4 system (v1.1 rule). The copy check compares each output with competitor product text from the price audit.

In [ ]:
from sentence_transformers import SentenceTransformer
retriever = g.Retriever(g.load_chunks())
guard = g.Guardrail(retriever, llm, MODEL, g.load_prompt(), prompt_version="guardrail_v1")
embedder = SentenceTransformer(g.EMBED_MODEL)
competitors = pd.read_csv("data/raw/price_audit/price_audit.csv")
comp_texts = (competitors["brand"] + " " + competitors["product"] + " " + competitors["fabric_composition"].fillna("")).tolist()

team = ca.ContentTeam(llm, guardrail=guard, embedder=embedder, competitor_texts=comp_texts)
requests = pd.read_csv(ca.REQUESTS).to_dict("records")
print(len(requests), "requests ·", len(team.claims), "approved claims:", [c["id"] for c in team.claims])

## 4. Run all 21 requests

In [ ]:
import time
outputs = []
t0 = time.time()
for i, req in enumerate(requests, 1):
    o = team.run(req)
    outputs.append(o)
    print(f"{i:2d}/{len(requests)} {o.request_id} {o.content_type:20s} passed={o.passed!s:5s} revisions={o.revisions} "
          f"critic={o.critic.get('overall','?')} {o.seconds:5.0f}s")
print(f"total {(time.time()-t0)/60:.1f} min")
ca.save_queue(outputs); ca.save_trace(outputs, ca.REPO / "ai" / "eval" / "content_trace.jsonl")
!cp ai/eval/content_approval_queue.csv ai/eval/content_trace.jsonl ai/logs/calls.csv "{DRIVE_OUT}/" && ls "{DRIVE_OUT}"

## 5. Guardrail pass test (automatic)

In [ ]:
q = pd.read_csv(ca.QUEUE)
m = ca.evaluate(q.to_dict("records"))
display(pd.Series(m))
display(q[["request_id", "content_type", "passed_checks", "revisions", "critic_overall", "open_issues"]])

## 6. Your scoring (the human step)
1. Open `ai_outputs/content_approval_queue.csv` in Google Sheets.
2. For every row, read `final_text` and fill in:
   - `your_voice_score`: 1–5 (5 = publish as is, 4 = minor edits, 3 = needs rework, 2 = off-brand, 1 = unusable). **Score before looking at the `critic_*` columns**, or hide them, so the agreement test is fair.
   - `your_decision`: approve / edit / reject.
   - `your_edit`: your edited text, if you changed it.
3. Download as CSV and save it to Drive as `ai_outputs/content_approval_queue_scored.csv`, then run the next cell.

In [ ]:
scored = pd.read_csv(f"{DRIVE_OUT}/content_approval_queue_scored.csv")
m2 = ca.evaluate(scored.to_dict("records"))
display(pd.Series(m2))
print(scored["your_decision"].value_counts())

## Answer
*One paragraph once scored: guardrail pass rate and mean revisions; share of outputs you scored ≥ 4; judge–human agreement (exact, within 1, weighted κ) and whether the judge is more lenient than you; the most common issues the checks caught; what you would change.*